In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

# 1. Load environment variables from .env file
load_dotenv()

# 2. Initialize the Groq model
#    You can adjust parameters like temperature and max_tokens as needed.
model = ChatGroq(
    model="llama-3.1-8b-instant",  # You can specify other models like mixtral-8x7b-32768
    temperature=0.0,
    max_retries=2,
)

# 3. Prepare a list of messages (system prompt and user question)
messages = [
    ("system", "You are a helpful AI assistant specialized in RAG pipelines."),
    ("human", "What is the main benefit of using RAG to reduce hallucinations?"),
]

# 4. Invoke the model and print the response
response = model.invoke(messages)
print(response.content)

/home/roman/Desktop/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAG (Reformer-based Augmented Generator) is a type of pipeline that has been shown to be effective in reducing hallucinations in various NLP tasks. The main benefit of using RAG to reduce hallucinations is its ability to leverage external knowledge sources, such as large-scale knowledge graphs or databases, to provide context and information that can help the model generate more accurate and relevant responses.

By incorporating external knowledge, RAG can reduce the likelihood of hallucinations, which occur when the model generates information that is not supported by the input or context. This is because RAG can draw upon a vast amount of knowledge and information to inform its responses, rather than relying solely on the input data.

Some of the key benefits of using RAG to reduce hallucinations include:

1. **Improved accuracy**: By leveraging external knowledge, RAG can generate more accurate and relevant responses, reducing the likelihood of hallucinations.
2. **Increased context

In [ ]:
import os
from dotenv import load_dotenv
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# Load environment variables
load_dotenv()

# ============================================================
# 1. Initialize ChromaDB client and collection
# ============================================================
CHROMA_PATH = os.getenv("CHROMA_PATH", "./chroma_db")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "wikipedia_articles")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

# ============================================================
# 2. Embedding model for query (use same as when you inserted docs)
# ============================================================
# If you used Chroma's default embedding function (all-MiniLM-L6-v2), 
# you can use the same model here.
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def get_query_embedding(query: str):
    """Generate embedding for the user question."""
    return embedder.encode(query).tolist()

# ============================================================
# 3. Retrieve and rank documents by cosine similarity
# ============================================================
def retrieve_relevant_docs(query: str, top_k: int = 5, similarity_threshold: float = 0.5):
    """
    Retrieve documents from ChromaDB that are relevant to the query.
    ChromaDB returns results sorted by distance (lowest = most similar).
    If the collection uses cosine distance, then distance = 1 - cosine_similarity.
    We'll convert distance to cosine similarity for thresholding.
    """
    query_embedding = get_query_embedding(query)
    
    # Query ChromaDB – it returns distances (cosine distance if collection was created with cosine)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    
    if not results['documents'] or not results['documents'][0]:
        return []
    
    docs = results['documents'][0]
    distances = results['distances'][0]   # these are cosine distances (0 = identical, 2 = opposite)
    metadatas = results['metadatas'][0] if results['metadatas'] else [{}] * len(docs)
    
    # Convert distance to cosine similarity: similarity = 1 - distance (for cosine distance)
    similarities = [1 - d for d in distances]
    
    # Filter by threshold
    relevant = []
    for doc, sim, meta in zip(docs, similarities, metadatas):
        if sim >= similarity_threshold:
            relevant.append({
                "content": doc,
                "similarity": sim,
                "metadata": meta
            })
    
    return relevant

# ============================================================
# 4. Build prompt with retrieved context
# ============================================================
def build_prompt(query: str, relevant_docs: list) -> str:
    """Create a prompt that includes the retrieved documents as context."""
    if not relevant_docs:
        return f"Question: {query}\n\nNo relevant documents found. Answer based on your general knowledge."
    
    context_parts = []
    for i, doc in enumerate(relevant_docs, 1):
        context_parts.append(f"Document {i}:\n{doc['content']}\n")
    
    context = "\n".join(context_parts)
    prompt = f"""You are a helpful assistant that answers questions strictly based on the provided context.
If the context does not contain enough information, say so clearly.

Context:
{context}

Question: {query}

Answer:"""
    return prompt

# ============================================================
# 5. Answer using Groq (LangChain)
# ============================================================
def answer_with_groq(question: str, relevant_docs: list) -> str:
    """Send the question + context to Groq and return the answer."""
    # Initialize Groq model
    llm = ChatGroq(
        model="llama-3.1-8b-instant",   # or "mixtral-8x7b-32768", "gemma2-9b-it"
        temperature=0.0,
        max_retries=2,
    )
    
    prompt = build_prompt(question, relevant_docs)
    
    messages = [
        SystemMessage(content="You are a precise, context-faithful assistant."),
        HumanMessage(content=prompt)
    ]
    
    response = llm.invoke(messages)
    return response.content

# ============================================================
# 6. Full pipeline
# ============================================================
def rag_with_chroma_groq(question: str, top_k: int = 5, similarity_threshold: float = 0.5):
    """Main function: retrieve docs, check relevance, generate answer."""
    print(f"📝 Question: {question}")
    relevant_docs = retrieve_relevant_docs(question, top_k, similarity_threshold)
    
    if not relevant_docs:
        print("⚠️ No documents passed the similarity threshold. Falling back to general knowledge.")
    else:
        print(f"✅ Retrieved {len(relevant_docs)} relevant documents.")
        for i, doc in enumerate(relevant_docs, 1):
            print(f"   Doc {i}: similarity = {doc['similarity']:.3f}")
    
    answer = answer_with_groq(question, relevant_docs)
    return answer


# Example: assuming your ChromaDB already has some documents
question = "What is Retrieval-Augmented Generation?"
answer = rag_with_chroma_groq(question, top_k=3, similarity_threshold=0.6)
print("\n💡 Final Answer:\n", answer)

/home/roman/Desktop/RAG/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2245.58it/s]


📝 Question: What is Retrieval-Augmented Generation?
✅ Retrieved 1 relevant documents.
   Doc 1: similarity = 0.615

💡 Final Answer:
 Retrieval-Augmented Generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources.
